# Fine-tuning vs RAG — Cuándo usar cada uno

Comparativa práctica: implementamos ambos enfoques para resolver el mismo problema
y analizamos coste, latencia, mantenimiento y calidad.

In [ ]:
import anthropic, os, json, time
import numpy as np

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('Cliente listo')

## 1. El problema: asistente de soporte técnico especializado

In [ ]:
# Base de conocimiento de soporte técnico (sería más grande en producción)
BASE_CONOCIMIENTO = [
    {
        'id': 'kb-001',
        'titulo': 'Error de conexión a la base de datos',
        'contenido': 'Error DB-500: Verifica que POSTGRES_URL esté configurada en .env. '
                     'Puerto por defecto: 5432. Si usas Docker, asegúrate de que el contenedor '
                     'está en la misma red. Comando de diagnóstico: pg_isready -h localhost -p 5432',
        'categoria': 'base_datos',
    },
    {
        'id': 'kb-002',
        'titulo': 'Límite de peticiones excedido',
        'contenido': 'Error 429: Has superado el límite de 1000 peticiones/hora en el plan básico. '
                     'Soluciones: (1) espera al siguiente ciclo horario, (2) actualiza al plan Pro '
                     'para 10.000 peticiones/hora, (3) implementa caché local para consultas repetidas.',
        'categoria': 'api',
    },
    {
        'id': 'kb-003',
        'titulo': 'Instalación en Windows',
        'contenido': 'En Windows usa: pip install acme-sdk[windows]. Requiere Visual C++ Build Tools. '
                     'Si falla, instala primero: winget install Microsoft.VisualStudio.2022.BuildTools. '
                     'WSL2 es la alternativa recomendada para desarrollo.',
        'categoria': 'instalacion',
    },
    {
        'id': 'kb-004',
        'titulo': 'Autenticación JWT inválida',
        'contenido': 'Error AUTH-401: El token JWT ha expirado o es inválido. '
                     'Los tokens expiran a las 24h por defecto. Renueva con POST /auth/refresh. '
                     'Verifica que ACME_SECRET_KEY sea la misma en todos los servidores.',
        'categoria': 'autenticacion',
    },
    {
        'id': 'kb-005',
        'titulo': 'Memoria insuficiente al procesar archivos grandes',
        'contenido': 'Para archivos >100MB usa el modo streaming: client.upload_stream(file_path). '
                     'Aumenta el límite de memoria con ACME_MAX_MEMORY=2048 (en MB). '
                     'Los archivos se procesan en chunks de 10MB automáticamente.',
        'categoria': 'rendimiento',
    },
]

print(f'Base de conocimiento: {len(BASE_CONOCIMIENTO)} artículos')

## 2. Enfoque A — Sin contexto (modelo base)

In [ ]:
def responder_sin_contexto(pregunta: str) -> dict:
    inicio = time.time()
    resp = client.messages.create(
        model=MODELO, max_tokens=200, temperature=0,
        system='Eres un asistente de soporte técnico de Acme SDK.',
        messages=[{'role': 'user', 'content': pregunta}]
    )
    return {
        'respuesta': resp.content[0].text,
        'latencia_ms': (time.time() - inicio) * 1000,
        'tokens': resp.usage.input_tokens + resp.usage.output_tokens,
        'metodo': 'sin_contexto',
    }

pregunta = '¿Por qué me da error 429 y cómo lo soluciono?'
resultado = responder_sin_contexto(pregunta)
print('SIN CONTEXTO:')
print(resultado['respuesta'])
print(f'\nLatencia: {resultado["latencia_ms"]:.0f}ms | Tokens: {resultado["tokens"]}')

## 3. Enfoque B — RAG (Retrieval-Augmented Generation)

In [ ]:
def obtener_embedding_simple(texto: str) -> np.ndarray:
    """Embedding simplificado basado en frecuencia de palabras (demo).
    En producción: usar client.embeddings o sentence-transformers."""
    palabras_clave = [
        'error', 'base', 'datos', 'conexion', 'limite', 'peticiones', '429',
        'instalacion', 'windows', 'autenticacion', 'jwt', 'token', 'memoria',
        'archivo', 'streaming', 'api', 'docker', 'postgres',
    ]
    texto_lower = texto.lower()
    return np.array([1.0 if p in texto_lower else 0.0 for p in palabras_clave])

# Indexar la base de conocimiento
embeddings_kb = [
    obtener_embedding_simple(doc['titulo'] + ' ' + doc['contenido'])
    for doc in BASE_CONOCIMIENTO
]

def buscar_documentos(pregunta: str, top_k: int = 2) -> list[dict]:
    emb_query = obtener_embedding_simple(pregunta)
    norma_query = np.linalg.norm(emb_query)
    if norma_query == 0:
        return BASE_CONOCIMIENTO[:top_k]

    scores = []
    for i, emb in enumerate(embeddings_kb):
        norma = np.linalg.norm(emb)
        if norma == 0:
            scores.append(0.0)
        else:
            scores.append(float(np.dot(emb_query, emb) / (norma_query * norma)))

    indices = np.argsort(scores)[-top_k:][::-1]
    return [BASE_CONOCIMIENTO[i] for i in indices if scores[i] > 0]

def responder_con_rag(pregunta: str) -> dict:
    inicio = time.time()

    docs_relevantes = buscar_documentos(pregunta)
    contexto = '\n\n'.join(
        f'[{d["id"]}] {d["titulo"]}:\n{d["contenido"]}'
        for d in docs_relevantes
    )

    resp = client.messages.create(
        model=MODELO, max_tokens=250, temperature=0,
        system='Eres un asistente de soporte técnico de Acme SDK. '
               'Responde SOLO usando la información del contexto proporcionado. '
               'Si el contexto no contiene la respuesta, dilo claramente.',
        messages=[{
            'role': 'user',
            'content': f'Contexto de la base de conocimiento:\n{contexto}\n\nPregunta: {pregunta}'
        }]
    )
    return {
        'respuesta': resp.content[0].text,
        'documentos_usados': [d['id'] for d in docs_relevantes],
        'latencia_ms': (time.time() - inicio) * 1000,
        'tokens': resp.usage.input_tokens + resp.usage.output_tokens,
        'metodo': 'rag',
    }

resultado_rag = responder_con_rag(pregunta)
print('CON RAG:')
print(resultado_rag['respuesta'])
print(f'\nDocumentos usados: {resultado_rag["documentos_usados"]}')
print(f'Latencia: {resultado_rag["latencia_ms"]:.0f}ms | Tokens: {resultado_rag["tokens"]}')

## 4. Enfoque C — Fine-tuning (preparación del dataset)

In [ ]:
# El fine-tuning requiere primero generar un dataset de pares pregunta-respuesta
# Usamos Claude para generar el dataset de forma automática

def generar_ejemplos_finetune(articulo: dict, n: int = 3) -> list[dict]:
    resp = client.messages.create(
        model=MODELO, max_tokens=500, temperature=0.7,
        messages=[{'role': 'user', 'content': f'''Genera {n} pares pregunta-respuesta de soporte técnico
basados en este artículo. Formato JSON:
[{{
  "pregunta": "...",
  "respuesta": "..."
}}]

Artículo: {articulo["titulo"]}
{articulo["contenido"]}'''}]
    )
    texto = resp.content[0].text
    if '```' in texto:
        texto = texto.split('```')[1].replace('json', '').strip()
    return json.loads(texto)

# Generar dataset para el primer artículo
ejemplos = generar_ejemplos_finetune(BASE_CONOCIMIENTO[1])  # error 429

print('Dataset generado para fine-tuning:')
for ej in ejemplos:
    print(f'  Q: {ej["pregunta"]}')
    print(f'  A: {ej["respuesta"][:100]}...')
    print()

# Convertir a formato JSONL para fine-tuning
dataset_ft = [{
    'messages': [
        {'role': 'system', 'content': 'Eres un asistente de soporte técnico de Acme SDK.'},
        {'role': 'user', 'content': ej['pregunta']},
        {'role': 'assistant', 'content': ej['respuesta']},
    ]
} for ej in ejemplos]

with open('/tmp/dataset_ft_acme.jsonl', 'w') as f:
    for item in dataset_ft:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Dataset JSONL guardado: {len(dataset_ft)} ejemplos')

## 5. Comparativa y árbol de decisión

In [ ]:
from IPython.display import display, HTML

comparativa = """
<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>
<tr style='background:#1565C0;color:white'>
  <th style='padding:10px'>Criterio</th>
  <th style='padding:10px'>RAG</th>
  <th style='padding:10px'>Fine-tuning</th>
  <th style='padding:10px'>Modelo base</th>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px;font-weight:bold'>Conocimiento actualizable</td>
  <td style='padding:8px;color:green'>✅ En tiempo real</td>
  <td style='padding:8px;color:red'>❌ Requiere reentrenamiento</td>
  <td style='padding:8px;color:red'>❌ Fecha corte</td>
</tr>
<tr>
  <td style='padding:8px;font-weight:bold'>Coste de setup</td>
  <td style='padding:8px;color:green'>✅ Bajo</td>
  <td style='padding:8px;color:orange'>⚠️ Alto (GPU + tiempo)</td>
  <td style='padding:8px;color:green'>✅ Ninguno</td>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px;font-weight:bold'>Precisión en dominio</td>
  <td style='padding:8px;color:green'>✅ Alta (si retrieval es bueno)</td>
  <td style='padding:8px;color:green'>✅ Muy alta</td>
  <td style='padding:8px;color:red'>❌ Baja (sin contexto)</td>
</tr>
<tr>
  <td style='padding:8px;font-weight:bold'>Latencia por llamada</td>
  <td style='padding:8px;color:orange'>⚠️ Media (+retrieval)</td>
  <td style='padding:8px;color:green'>✅ Baja</td>
  <td style='padding:8px;color:green'>✅ Baja</td>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px;font-weight:bold'>Trazabilidad</td>
  <td style='padding:8px;color:green'>✅ Cita fuentes</td>
  <td style='padding:8px;color:red'>❌ Caja negra</td>
  <td style='padding:8px;color:red'>❌ Caja negra</td>
</tr>
<tr>
  <td style='padding:8px;font-weight:bold'>Mejor para</td>
  <td style='padding:8px'>Docs que cambian, compliance, trazabilidad</td>
  <td style='padding:8px'>Tono/estilo específico, miles de llamadas/día</td>
  <td style='padding:8px'>Tareas generales sin contexto propio</td>
</tr>
</table>
"""
display(HTML(comparativa))

In [ ]:
# Árbol de decisión: ¿RAG o fine-tuning?
def decidir_estrategia(contexto: dict) -> str:
    if contexto.get('datos_cambian_frecuentemente'):
        return 'RAG — Los datos cambian: actualiza el índice sin reentrenar'
    if contexto.get('necesita_citar_fuentes'):
        return 'RAG — La trazabilidad es un requisito'
    if contexto.get('volumen_llamadas_dia', 0) > 10_000 and not contexto.get('contexto_largo'):
        return 'Fine-tuning — Alto volumen + respuestas cortas = amortizas el coste'
    if contexto.get('estilo_muy_especifico'):
        return 'Fine-tuning — El tono/formato específico es difícil de lograr con prompts'
    if contexto.get('base_conocimiento_grande'):
        return 'RAG — Una KB grande no cabe en el contexto, necesitas retrieval'
    return 'RAG — Es el punto de partida más seguro y más fácil de mantener'

casos = [
    {'nombre': 'Chatbot de soporte con docs actualizados semanalmente',
     'datos_cambian_frecuentemente': True},
    {'nombre': 'Asistente que genera informes en el estilo de tu empresa',
     'estilo_muy_especifico': True, 'volumen_llamadas_dia': 50_000},
    {'nombre': 'QA sobre documentación legal (auditoría)',
     'necesita_citar_fuentes': True},
    {'nombre': 'Clasificador de tickets (100k/día, respuesta: 1 palabra)',
     'volumen_llamadas_dia': 100_000, 'estilo_muy_especifico': True},
]

for caso in casos:
    nombre = caso.pop('nombre')
    print(f'Caso: {nombre}')
    print(f'  → {decidir_estrategia(caso)}\n')